# Task 2: End-to-End ML Pipeline with Scikit-learn
**Internship:** DevelopersHub Corporation – AI/ML Engineering (Advanced)

## Objective
Build a reusable, production-ready machine learning pipeline to predict customer churn, including preprocessing, hyperparameter tuning, and model export.

## Dataset
**Telco Customer Churn Dataset** — available on Kaggle
- ~7,000 customers, 21 features
- Target: `Churn` (Yes/No)

**Kaggle Link:** https://www.kaggle.com/datasets/blastchar/telco-customer-churn

## Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve
)

import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.dpi'] = 100

print('Libraries imported!')

## Step 2: Load the Dataset

In [ ]:
try:
    df = pd.read_csv('/kaggle/input/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv')
    print('Loaded from Kaggle input.')
except FileNotFoundError:
    df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
    print('Loaded from current directory.')

print(f'Shape: {df.shape}')
df.head()

## Step 3: Data Cleaning & Preprocessing

In [ ]:
# Drop customer ID (not predictive)
df = df.drop('customerID', axis=1)

# TotalCharges has some blank strings - convert to numeric
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

print('Missing values after cleaning:')
print(df.isnull().sum()[df.isnull().sum() > 0])

# Encode target variable
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

print(f'\nFinal shape: {df.shape}')
print(f'Churn rate: {df["Churn"].mean():.2%}')

In [ ]:
# Identify numeric vs categorical columns
numeric_features = ['tenure', 'MonthlyCharges', 'TotalCharges']
categorical_features = [col for col in df.columns
                         if col not in numeric_features + ['Churn']]

print(f'Numeric features ({len(numeric_features)}): {numeric_features}')
print(f'\nCategorical features ({len(categorical_features)}): {categorical_features}')

## Step 4: Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Churn distribution
churn_counts = df['Churn'].value_counts()
axes[0].pie(churn_counts, labels=['No Churn', 'Churn'], colors=['#2ECC71', '#E74C3C'],
            autopct='%1.1f%%', startangle=140, textprops={'fontsize': 12})
axes[0].set_title('Customer Churn Distribution', fontsize=14, fontweight='bold')

# Tenure vs Churn
sns.boxplot(data=df, x='Churn', y='tenure', palette={0: '#2ECC71', 1: '#E74C3C'}, ax=axes[1])
axes[1].set_xticklabels(['No Churn', 'Churn'])
axes[1].set_title('Tenure by Churn Status', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Churn Status', fontsize=12)
axes[1].set_ylabel('Tenure (Months)', fontsize=12)

plt.tight_layout()
plt.show()

print('Insight: Customers who churn tend to have lower tenure (newer customers).')

In [ ]:
# Monthly charges and contract type vs churn
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df, x='Churn', y='MonthlyCharges',
            palette={0: '#2ECC71', 1: '#E74C3C'}, ax=axes[0])
axes[0].set_xticklabels(['No Churn', 'Churn'])
axes[0].set_title('Monthly Charges by Churn Status', fontsize=14, fontweight='bold')

churn_by_contract = df.groupby('Contract')['Churn'].mean().sort_values(ascending=False)
churn_by_contract.plot(kind='bar', color='#E74C3C', ax=axes[1], edgecolor='white')
axes[1].set_title('Churn Rate by Contract Type', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Churn Rate', fontsize=12)
axes[1].set_xticklabels(churn_by_contract.index, rotation=20, ha='right')

plt.tight_layout()
plt.show()

print('Insight: Month-to-month contracts have a MUCH higher churn rate than yearly contracts.')

## Step 5: Build the Preprocessing Pipeline

In [ ]:
# Numeric pipeline: impute missing values, then scale
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical pipeline: impute missing, then one-hot encode
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Combine into a single ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

print('Preprocessing pipeline built!')
print(f'  Numeric pipeline: Impute (median) -> Scale')
print(f'  Categorical pipeline: Impute (mode) -> One-Hot Encode')

## Step 6: Train/Test Split

In [ ]:
X = df.drop('Churn', axis=1)
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training samples: {X_train.shape[0]}')
print(f'Testing samples : {X_test.shape[0]}')

## Step 7: Build Full Pipelines (Preprocessing + Model)

In [ ]:
# ----- Pipeline 1: Logistic Regression -----
lr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42, max_iter=1000))
])

# ----- Pipeline 2: Random Forest -----
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

print('Full pipelines created (preprocessing + model in one object)!')
print('This means new raw data can be passed directly without manual preprocessing.')

## Step 8: Hyperparameter Tuning with GridSearchCV

In [ ]:
# Logistic Regression grid search
lr_param_grid = {
    'classifier__C': [0.01, 0.1, 1, 10],
    'classifier__penalty': ['l2'],
    'classifier__solver': ['lbfgs']
}

lr_grid = GridSearchCV(
    lr_pipeline, lr_param_grid,
    cv=5, scoring='roc_auc', n_jobs=-1, verbose=1
)

print('Running GridSearchCV for Logistic Regression...')
lr_grid.fit(X_train, y_train)

print(f'\nBest params: {lr_grid.best_params_}')
print(f'Best CV AUC-ROC: {lr_grid.best_score_:.4f}')

In [ ]:
# Random Forest grid search
rf_param_grid = {
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth': [5, 10, None],
    'classifier__min_samples_split': [2, 5]
}

rf_grid = GridSearchCV(
    rf_pipeline, rf_param_grid,
    cv=5, scoring='roc_auc', n_jobs=-1, verbose=1
)

print('Running GridSearchCV for Random Forest...')
rf_grid.fit(X_train, y_train)

print(f'\nBest params: {rf_grid.best_params_}')
print(f'Best CV AUC-ROC: {rf_grid.best_score_:.4f}')

## Step 9: Final Evaluation on Test Set

In [ ]:
best_lr = lr_grid.best_estimator_
best_rf = rf_grid.best_estimator_

results = []
for name, model in [('Logistic Regression', best_lr), ('Random Forest', best_rf)]:
    preds = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    
    acc = accuracy_score(y_test, preds)
    auc = roc_auc_score(y_test, proba)
    
    results.append({'Model': name, 'Accuracy': acc, 'AUC-ROC': auc})
    print(f'{name}: Accuracy={acc:.4f}, AUC-ROC={auc:.4f}')

results_df = pd.DataFrame(results)
results_df

In [ ]:
# Confusion matrices & ROC curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (name, model) in zip(axes, [('Logistic Regression', best_lr), ('Random Forest', best_rf)]):
    preds = model.predict(X_test)
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['No Churn', 'Churn'],
                yticklabels=['No Churn', 'Churn'], ax=ax)
    ax.set_title(f'{name} - Confusion Matrix', fontsize=13, fontweight='bold')
    ax.set_xlabel('Predicted', fontsize=11)
    ax.set_ylabel('Actual', fontsize=11)

plt.tight_layout()
plt.show()

# ROC Curve comparison
plt.figure(figsize=(9, 7))
for name, model in [('Logistic Regression', best_lr), ('Random Forest', best_rf)]:
    proba = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc_val = roc_auc_score(y_test, proba)
    plt.plot(fpr, tpr, lw=2, label=f'{name} (AUC = {auc_val:.3f})')

plt.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random Classifier')
plt.xlabel('False Positive Rate', fontsize=13)
plt.ylabel('True Positive Rate', fontsize=13)
plt.title('ROC Curve Comparison', fontsize=15, fontweight='bold')
plt.legend(fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Detailed classification reports
print('=== Logistic Regression ===')
print(classification_report(y_test, best_lr.predict(X_test), target_names=['No Churn', 'Churn']))

print('=== Random Forest ===')
print(classification_report(y_test, best_rf.predict(X_test), target_names=['No Churn', 'Churn']))

## Step 10: Export the Pipeline with Joblib

In [ ]:
# Choose the best performing pipeline based on AUC-ROC
best_model_name = results_df.loc[results_df['AUC-ROC'].idxmax(), 'Model']
best_pipeline = best_lr if best_model_name == 'Logistic Regression' else best_rf

print(f'Best model: {best_model_name}')

# Export the ENTIRE pipeline (preprocessing + model) as a single file
joblib.dump(best_pipeline, 'churn_prediction_pipeline.joblib')

print('Pipeline exported to: churn_prediction_pipeline.joblib')
print('\nThis single file can be loaded anywhere and used directly on raw data:')
print('  loaded_pipeline = joblib.load("churn_prediction_pipeline.joblib")')
print('  predictions = loaded_pipeline.predict(new_raw_data)')

In [ ]:
# Verify the saved pipeline works correctly
loaded_pipeline = joblib.load('churn_prediction_pipeline.joblib')
test_prediction = loaded_pipeline.predict(X_test.iloc[:5])
test_actual = y_test.iloc[:5].values

print('Verification - Loaded pipeline predictions on 5 samples:')
print(f'  Predicted: {test_prediction}')
print(f'  Actual   : {test_actual}')
print('\nPipeline loads and predicts correctly!')

## Step 11: Final Summary

### Approach
1. Cleaned the Telco Churn dataset (handled missing `TotalCharges` values)
2. Built a `ColumnTransformer` to scale numeric features and one-hot encode categoricals
3. Wrapped preprocessing + model into a single `sklearn.Pipeline` (production-ready)
4. Tuned hyperparameters for both Logistic Regression and Random Forest using `GridSearchCV` (5-fold CV)
5. Evaluated on a held-out test set using Accuracy, AUC-ROC, Confusion Matrix
6. Exported the best pipeline with `joblib` for reuse in production

### Key Results
- Both models achieved strong AUC-ROC scores (see results table above)
- **Month-to-month contracts** and **low tenure** are the strongest churn predictors
- The exported `.joblib` pipeline can take **raw, unprocessed customer data** and output predictions directly — no manual preprocessing required
- This is a **production-ready** approach: the same pipeline used for training can be deployed as-is

In [ ]:
print('=' * 55)
print('ADVANCED TASK 2 COMPLETE – Churn Prediction Pipeline')
print('=' * 55)
print(f'Dataset       : Telco Customer Churn ({df.shape[0]} customers)')
print(f'Best Model    : {best_model_name}')
print(f'Best AUC-ROC  : {results_df["AUC-ROC"].max():.4f}')
print(f'Exported File : churn_prediction_pipeline.joblib')
print('Pipeline includes: Imputation -> Scaling/Encoding -> Trained Classifier')